# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory","24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Claro/home")
_files_claro = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*PIT/*.gz") if file.is_file()]
df = pd.DataFrame(_files_claro, columns=["file_path","parent_folder"])
df["exchange_id"] = df["parent_folder"].str.slice(stop=4)
files_claro = df.groupby("exchange_id").sample(15,random_state=42)["file_path"].to_list()
files_claro[:5]

In [ ]:
# files_claro = [file for file,_ in _files_claro]

### Arquivos intermediário e final

In [ ]:
extracted_claro ="/assets/data/cdr/teletools/db/extraidos/pit_claro_extraido.parquet"
transformed_claro ="/assets/data/cdr/teletools/db/transformados/pit_claro_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType

stfc_pit_claro_schema = StructType(
    [
        StructField("recordType", StringType(), True),
        StructField("recordingEntity", StringType(), True),
        StructField("mscIncomingROUTE", StringType(), True),
        StructField("mscOutgoingROUTE", StringType(), True),
        StructField("callingNumber", StringType(), True),
        StructField("calledNumber", StringType(), True),
        StructField("isdnBasicService", StringType(), True),
        StructField("seizureTime", StringType(), True),
        StructField("answerTime", StringType(), True),
        StructField("releaseTime", StringType(), True),
        StructField("callDuration", StringType(), True),
        StructField("causeForTerm", StringType(), True),
        StructField("diagnostics_gsm0408", StringType(), True),
        StructField("diagnostics_gsm0902", StringType(), True),
        StructField("diagnostics_ccittQ767", StringType(), True),
        StructField("callReference", StringType(), True),
        StructField("sequenceNumber", StringType(), True),
        StructField("recordExtensions", StringType(), True),
        StructField("partialRecordType", StringType(), True),
        StructField("basicservice_bearerService", StringType(), True),
        StructField("basicservice_teleservice", StringType(), True),
        StructField("basicservice_isdnService", StringType(), True),
        StructField("additionalChgInfo_chargeIndicat", StringType(), True),
        StructField("additionalChgInfo_chargeParamet", StringType(), True),
        StructField("ussdCallBackFlag", StringType(), True),
        StructField("originalCalledNumber", StringType(), True),
        StructField("chargeAreaCode", StringType(), True),
        StructField("rateIndication", StringType(), True),
        StructField("roamingNumber", StringType(), True),
        StructField("mscOutgoingCircuit", StringType(), True),
        StructField("mscIncomingCircuit", StringType(), True),
        StructField("orgMSCId", StringType(), True),
        StructField("callEmlppPriority", StringType(), True),
        StructField("eaSubscriberInfo", StringType(), True),
        StructField("selectedCIC", StringType(), True),
        StructField("callerportedflag", StringType(), True),
        StructField("subscriberCategory", StringType(), True),
        StructField("cUGOutgoingAccessIndicator", StringType(), True),
        StructField("cUGInterlockCode", StringType(), True),
        StructField("cUGIncomingAccessUsed", StringType(), True),
        StructField("mscIncomingRouteAttribute", StringType(), True),
        StructField("mscOutgoingRouteAttribute", StringType(), True),
        StructField("networkCallReference", StringType(), True),
        StructField("setupTime", StringType(), True),
        StructField("alertingTime", StringType(), True),
        StructField("voiceIndicator", StringType(), True),
        StructField("bCategory", StringType(), True),
        StructField("callType", StringType(), True),
        StructField("chargePulseNum", StringType(), True),
        StructField("disconnectparty", StringType(), True),
        StructField("chargePulseNumforITXTXA", StringType(), True),
        StructField("networkOperatorId", StringType(), True),
        StructField("audioDataType", StringType(), True),
        StructField("recordNumber", StringType(), True),
        StructField("partyRelCause_releaseParty", StringType(), True),
        StructField("partyRelCause_releaseCause", StringType(), True),
        StructField("chargeLevel", StringType(), True),
        StructField("locationNum", StringType(), True),
        StructField("locationNumberNai", StringType(), True),
        StructField("translatedNumber", StringType(), True),
        StructField("cmnFlag", StringType(), True),
        StructField("icidvalue", StringType(), True),
        StructField("origioi", StringType(), True),
        StructField("termioi", StringType(), True),
        StructField("calledportedflag", StringType(), True),
        StructField("locationroutingnumber", StringType(), True),
        StructField("intermediateChargingInd", StringType(), True),
        StructField("mscOutgoingROUTENumber", StringType(), True),
        StructField("mscIncomingROUTENumber", StringType(), True),
        StructField("drcCallId", StringType(), True),
        StructField("drcCallRN", StringType(), True),
        StructField("nPDipIndicator", StringType(), True),
        StructField("aNSIRoutingNumber", StringType(), True),
        StructField("lRNSource", StringType(), True),
        StructField("wPSCallFlag", StringType(), True),
        StructField("redirectingNumber", StringType(), True),
        StructField("redirectingCounter", StringType(), True),
        StructField("officeName", StringType(), True),
        StructField("npa-Nxx", StringType(), True),
        StructField("globalCallReference", StringType(), True),
        StructField("callerIpInfo_origIpAddress", StringType(), True),
        StructField("callerIpInfo_termIpAddress", StringType(), True),
        StructField("callerIpInfo_passertedId", StringType(), True),
        StructField("callerIpInfo_usedCodec1", StringType(), True),
        StructField("callerIpInfo_usedCodec2", StringType(), True),
        StructField("callerIpInfo_usedCodec3", StringType(), True),
        StructField("callerIpInfo_usedCodec4", StringType(), True),
        StructField("callerIpInfo_usedCodec5", StringType(), True),
        StructField("callerIpInfo_usedCodec6", StringType(), True),
        StructField("callerIpInfo_usedCodec7", StringType(), True),
        StructField("callerIpInfo_usedCodec8", StringType(), True),
        StructField("callerIpInfo_originatingPort", StringType(), True),
        StructField("callerIpInfo_terminatingPort", StringType(), True),
        StructField("callerIpInfo_qosStatistics", StringType(), True),
        StructField("calledIpInfo_origIpAddress", StringType(), True),
        StructField("calledIpInfo_termIpAddress", StringType(), True),
        StructField("calledIpInfo_passertedId", StringType(), True),
        StructField("calledIpInfo_usedCodec1", StringType(), True),
        StructField("calledIpInfo_usedCodec2", StringType(), True),
        StructField("calledIpInfo_usedCodec3", StringType(), True),
        StructField("calledIpInfo_usedCodec4", StringType(), True),
        StructField("calledIpInfo_usedCodec5", StringType(), True),
        StructField("calledIpInfo_usedCodec6", StringType(), True),
        StructField("calledIpInfo_usedCodec7", StringType(), True),
        StructField("calledIpInfo_usedCodec8", StringType(), True),
        StructField("calledIpInfo_originatingPort", StringType(), True),
        StructField("calledIpInfo_terminatingPort", StringType(), True),
        StructField("calledIpInfo_qosStatistics", StringType(), True),
        StructField("OriginatingOperator", StringType(), True),
        StructField("TerminatingOperator", StringType(), True),
    ]
)

In [ ]:
df_raw = spark.read.csv(files_claro, sep=",", schema=stfc_pit_claro_schema)
df_raw = df_raw.filter(F.col("recordType")=="5")
df_raw.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_raw.groupBy("causeForTerm").count().show()

In [ ]:
df_raw.select("networkCallReference", "callType").show(5)

In [ ]:
column_indices = (1,2,3,4,5,9,10,11,42,43,108)
column_names = ("bilhetador","rota_entrada","rota_saida","numero_origem","numero_destino","_data_hora_fim","duracao","resultado_chamada","referencia","_data_hora_chamada", "_prestadora_origem")
columns_to_keep = [
    F.col(df_raw.columns[index]).alias(column_name)
    for index, column_name in zip(
        column_indices,
        column_names,
    )
]
df = df_raw.select(*columns_to_keep).withColumn("duracao", F.col("duracao").cast(T.IntegerType()))

df.show(5)

In [ ]:
df.printSchema()

In [ ]:
df.filter(F.col("duracao")>0).show(5)

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_claro, extracted_claro,"stfc_pit_claro")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5, truncate=False)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_pit_claro(extracted_claro, transformed_claro)

In [ ]:
df_transformed = spark.read.parquet(transformed_claro)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_transformed.groupBy("no_tipo_chamada").count().show()